# Gazetteers and Rule-Based Extraction

**Computational Text Analysis Pack · Module 15 — Rule-Based and Symbolic Methods**

*Technique:* ANNIE/JAPE-style entity extraction; when rules beat statistical NER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/04-rule-based-methods/15c_gazetteers_and_rules.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Named-entity recognition is usually presented as a machine-learning task. The
survey makes the case for the older approach in the situations DH actually
faces: **gazetteer-driven rules still beat statistical NER on domain-specific
entities**, and they do it transparently, on small data, with no training set.

That combination — no labelled data, unusual entities, need to justify every
decision — describes most humanities projects.

## The tooling

**GATE** with its **ANNIE** pipeline and **JAPE** (Java Annotation Patterns
Engine) is the reference implementation: gazetteer lists, then cascaded pattern
rules over annotations. It is still maintained and still used where rules win.
**spaCy** offers the same idea in Python through its `EntityRuler`,
`Matcher` and `PhraseMatcher` components, which is the modern way to combine
rules with a statistical model.

We build a small ANNIE in plain Python: a gazetteer, then rules over it.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import re

text_docs = df_colonial["text"].head(1500).tolist()

# ── Stage 1: the gazetteer ────────────────────────────────────────────────────
# In GATE these are plain lists, one entity per line, curated by the researcher.
GAZETTEER = {
    "TITLE": {"mr", "mrs", "dr", "sir", "lord", "lady", "captain", "colonel",
              "chief", "king", "queen", "emir", "sultan", "reverend", "governor"},
    "PLACE_TYPE": {"river", "town", "city", "village", "kingdom", "province",
                   "island", "coast", "mountain", "lake", "port"},
    "PEOPLE_WORD": {"tribe", "people", "natives", "chiefs", "warriors"},
}
for k, v in GAZETTEER.items():
    print(f"{k:<14}{len(v):>4} entries")

## Stage 2: JAPE-style rules over the gazetteer

A JAPE rule is a pattern over *annotations*, not characters — which is why the
gazetteer has to run first. Each rule below states its left context, what it
captures, and the label it assigns, exactly as a JAPE phase does.

In [ ]:
TITLE = "|".join(sorted(GAZETTEER["TITLE"]))
PTYPE = "|".join(sorted(GAZETTEER["PLACE_TYPE"]))

RULES = [
    ("PERSON", rf"\b(?:{TITLE})\.?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)",
     "a gazetteer title followed by capitalised words"),
    ("PLACE", rf"\b(?:{PTYPE})\s+of\s+([A-Z][a-z]+)",
     "a place-type word, 'of', then a capitalised word"),
    ("PLACE", r"\b(?:at|in|from|to|near)\s+([A-Z][a-z]{3,})",
     "a locative preposition followed by a capitalised word"),
    ("GROUP", rf"\b([A-Z][a-z]+)\s+(?:{'|'.join(GAZETTEER['PEOPLE_WORD'])})",
     "a capitalised word followed by a people-word"),
]

def annotate(docs):
    found = {label: Counter() for label, _, _ in RULES}
    for doc in docs:
        for label, pattern, _ in RULES:
            for m in re.finditer(pattern, doc):
                found[label][m.group(1)] += 1
    return found

ents = annotate(text_docs)
for label in ents:
    print(f"\n{label} — top matches")
    for name, n in ents[label].most_common(10):
        print(f"   {name:<22}{n:>5}")

In [ ]:
# Which rule fired for each hit? Provenance is the whole point of this approach.
print(f"{'label':<9}{'rule':<62}{'hits':>6}")
print("-" * 78)
for label, pattern, description in RULES:
    hits = sum(len(re.findall(pattern, d)) for d in text_docs)
    print(f"{label:<9}{description:<62}{hits:>6}")
print()
print("Every extracted entity can be traced to one line of this table. A")
print("statistical NER model gives you a confidence score instead, and no")
print("account of why. For a footnote, the rule is the better object.")

## Where rules beat learning, measured

The comparison the survey implies: a statistical classifier needs labelled
examples and generalises from them; a rule needs domain knowledge and
generalises not at all. On rare, domain-specific entities the rule wins, because
the classifier never saw enough examples to learn them.

In [ ]:
# Treat rule output as silver labels, then see how much data a classifier
# needs before it can recover the same distinctions.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

windows, labels = [], []
for doc in text_docs:
    for label, pattern, _ in RULES:
        for m in re.finditer(pattern, doc):
            s, e = max(0, m.start() - 60), min(len(doc), m.end() + 60)
            windows.append(doc[s:e])
            labels.append(label)

print(f"{len(windows)} labelled context windows from the rules")
if len(set(labels)) > 1 and len(windows) > 60:
    Xtr, Xte, ytr, yte = train_test_split(windows, labels, test_size=0.3,
                                          random_state=0, stratify=labels)
    print(f"\n{'training examples':>20}{'classifier accuracy':>22}")
    print("-" * 42)
    for frac in [0.05, 0.1, 0.25, 0.5, 1.0]:
        n = max(20, int(len(Xtr) * frac))
        vec = TfidfVectorizer(token_pattern=r"\w+", max_features=3000)
        A = vec.fit_transform(Xtr[:n])
        clf = LogisticRegression(max_iter=1000).fit(A, ytr[:n])
        acc = accuracy_score(yte, clf.predict(vec.transform(Xte)))
        print(f"{n:>20}{acc:>22.3f}")
    print()
    print("The rules had zero training examples and 100% agreement with")
    print("themselves. The classifier needs hundreds of examples to approach")
    print("what four lines of pattern already encoded.")

In [ ]:
# And the honest limit: rules do not transfer.
hausa = df[df["language"] == "Hausa"]["text"].head(300).tolist()
h_ents = annotate(hausa)
print("The same rule set applied to Hausa news:")
for label in h_ents:
    print(f"   {label:<9}{sum(h_ents[label].values()):>5} hits")
print()
print("Near-zero, because every rule encodes English word order and English")
print("capitalisation conventions. Rules are precise and completely")
print("non-transferable. That is the trade against a statistical model.")

### 🔧 Your turn 1

Add ten place names you know to a new gazetteer list and write a rule that uses it. How many new entities does it find?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

A specific gazetteer usually finds a lot at once, because in a corpus about a
region the same places recur constantly — this is precisely the case where GATE
beats a trained model, and why DH projects on a defined archive so often ship a
curated list. The corresponding cost is maintenance: the list is now part of
your method and has to be published with the results, or the extraction cannot
be reproduced.

</details>